## 1. Instalar dependências

In [ ]:
!pip install -q numpy pandas scikit-learn scipy tqdm

## 2. Montar o Google Drive

Os dados e resultados ficam no Drive. Rode uma vez por sessão e siga o fluxo de autorização do Google.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Imports

In [ ]:
import os
import time

import numpy as np
import pandas as pd
from scipy import sparse
from tqdm import tqdm

from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

## 4. DIRETÓRIOS

In [ ]:
# Raiz do projeto no Drive (AJUSTE AQUI)
BASE_DIR = "DIRETORIO_BASE"

BASE_DADOS = os.path.join(BASE_DIR, "dados")
BASE_SRC_RFE = os.path.join(BASE_DIR, "src_rfe")
os.makedirs(BASE_SRC_RFE, exist_ok=True)

CAMINHO_ARQUIVO = os.path.join(BASE_DADOS, "mh1m_balanceadas.npz")

ORC_INDIVIDUAIS = os.path.join(BASE_DADOS, "orcamento_shap_individuais.csv")
ORC_PERM_OPCODES = os.path.join(BASE_DADOS, "orcamento_shap_permissions_opcodes.csv")
ORC_TODAS = os.path.join(BASE_DADOS, "orcamento_shap_todas.csv")

# 5. ESTIMADOR

In [ ]:
STEP = 0.01  # remove 1% das features por iteracao

def novo_estimador():
    # liblinear: bom para alta dimensao e dados esparsos/binarios
    return LogisticRegression(solver="liblinear", max_iter=1000, random_state=42)

# Acumuladores de tempo
tempos_rfe = []
tempos_dataset = []

## 6. Carga do dataset original

In [ ]:
print("Carregando dataset original...")
dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)
X = dados["data"]
y = dados["classes"].astype(np.int8)
colunas = dados["column_names"]
print(f"Dataset original: X={X.shape}, y={y.shape}, colunas={colunas.shape}")

## 7. Funções auxiliares

In [ ]:
def indices_do_grupo(nome_grupo):
    if nome_grupo == "permissions_opcodes":
        idx = [i for i, n in enumerate(colunas) if n.startswith("permissions::") or n.startswith("opcodes::")]
    elif nome_grupo == "todas":
        idx = list(range(len(colunas)))
    else:
        idx = [i for i, n in enumerate(colunas) if n.startswith(f"{nome_grupo}::")]
    return idx

In [ ]:
def gera_ranking_rfe(nome_grupo, k_alvo):
    """Roda RFE no grupo ate sobrar k_alvo features. Usa matriz esparsa (dados binarios)
    para economizar memoria. Mede o tempo, salva o ranking e retorna as features selecionadas."""
    idx = indices_do_grupo(nome_grupo)
    # Matriz esparsa CSR a partir do subset (dados binarios 0/1 -> grande economia de RAM)
    X_grupo = sparse.csr_matrix(X[:, idx].astype(np.float32))
    colunas_grupo = colunas[idx]

    n_total = X_grupo.shape[1]
    k_alvo = int(min(k_alvo, n_total))

    print(f"\n[{nome_grupo}] RFE: {n_total} -> {k_alvo} features (step={STEP}). Pode demorar...")

    t0 = time.perf_counter()
    selector = RFE(
        estimator=novo_estimador(),
        n_features_to_select=k_alvo,
        step=STEP,
        verbose=1,   # progresso real a cada iteracao
    )
    selector.fit(X_grupo, y)
    tempo_rfe_s = time.perf_counter() - t0

    df_rank = pd.DataFrame({
        "feature": colunas_grupo,
        "rfe_ranking": selector.ranking_,
        "selecionada": selector.support_,
    }).sort_values(by="rfe_ranking", ascending=True).reset_index(drop=True)

    pasta = os.path.join(BASE_SRC_RFE, nome_grupo)
    os.makedirs(pasta, exist_ok=True)
    df_rank.to_csv(os.path.join(pasta, "ranking_features_rfe.csv"), index=False)

    selecionadas = colunas_grupo[selector.support_].tolist()

    tempos_rfe.append({
        "grupo": nome_grupo,
        "n_features_total": n_total,
        "k_alvo": k_alvo,
        "n_selecionadas": len(selecionadas),
        "step": STEP,
        "tempo_rfe_s": tempo_rfe_s,
    })
    # salva os tempos a cada grupo (checkpoint, caso interrompa)
    pd.DataFrame(tempos_rfe).to_csv(os.path.join(BASE_SRC_RFE, "tempos_rfe.csv"), index=False)

    print(f"[{nome_grupo}] RFE concluido em {tempo_rfe_s:.2f}s | {len(selecionadas)} features")
    return selecionadas

In [ ]:
def monta_e_salva_dataset(nomes_features, caminho_saida, rotulo):
    t0 = time.perf_counter()
    nomes_arr = np.array(list(dict.fromkeys(nomes_features)))
    existe = np.isin(nomes_arr, colunas)
    if not existe.all():
        faltando = nomes_arr[~existe]
        print(f"  ATENCAO [{rotulo}]: {len(faltando)} feature(s) nao encontrada(s). Ex.: {faltando[:5]}")

    mask = np.isin(colunas, nomes_arr)
    X_red = X[:, mask]
    colunas_red = colunas[mask]
    np.savez_compressed(caminho_saida, data=X_red, classes=y, column_names=colunas_red)
    tempo_dataset_s = time.perf_counter() - t0

    print(f"  [{rotulo}] data={X_red.shape}, column_names={colunas_red.shape}")
    print(f"  [{rotulo}] salvo em: {caminho_saida} | tempo={tempo_dataset_s:.4f}s")
    chk = np.load(caminho_saida, allow_pickle=True)
    print(f"  [{rotulo}] verificacao -> data={chk['data'].shape}, classes={chk['classes'].shape}, column_names={chk['column_names'].shape}")

    tempos_dataset.append({
        "dataset": rotulo, "caminho": caminho_saida,
        "n_features": int(X_red.shape[1]), "tempo_montagem_s": tempo_dataset_s,
    })
    pd.DataFrame(tempos_dataset).to_csv(os.path.join(BASE_SRC_RFE, "tempos_montagem_datasets_rfe.csv"), index=False)

## 8. Dataset 1 — Individuais (união)

In [ ]:
def roda_dataset1():
    print("\n=== DATASET 1: individuais (uniao) ===")
    orc_ind = pd.read_csv(ORC_INDIVIDUAIS)
    k_por_grupo = dict(zip(orc_ind["grupo"], orc_ind["n_features_selecionadas"]))
    print("K por grupo (SHAP):", k_por_grupo)

    features_uniao = []
    for nome_grupo in tqdm(["intents", "permissions", "opcodes", "apicalls"], desc="DS1 grupos"):
        k = int(k_por_grupo[nome_grupo])
        sel = gera_ranking_rfe(nome_grupo, k)
        features_uniao.extend(sel)

    monta_e_salva_dataset(features_uniao,
                          os.path.join(BASE_DADOS, "mh1m_balanceadas_rfe.npz"),
                          "DS1 rfe individuais")

## 11. Execução

Rode os datasets na ordem desejada. Comente os que já terminou para não repetir.

In [ ]:
roda_dataset1()

print("\n=== Concluido ===")
print("Tempos em:", os.path.join(BASE_SRC_RFE, "tempos_rfe.csv"))